# LeetCode 1049 — Last Stone Weight II bằng Python

Link đề: https://leetcode.com/problems/last-stone-weight-ii/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Dynamic Programming (quy hoạch động)

Skeleton LeetCode:

```python
class Solution:
    def lastStoneWeightII(self, stones: List[int]) -> int:
        ...
```

Ghi chú: ta chơi một trò với các viên đá: mỗi lượt chọn **hai** viên bất kỳ có khối lượng `x <= y` rồi đập vào nhau: nếu `x == y` cả hai biến mất, nếu `x != y` viên nhẹ `x` biến mất còn viên nặng còn lại có khối lượng `y - x`. Cuối cùng còn tối đa một viên; hãy tìm khối lượng **nhỏ nhất** có thể của viên còn lại (`0` nếu không còn viên nào). Điểm mấu chốt (không hiển nhiên): kết quả luôn có dạng `|±s_1 ± s_2 ± ... ± s_n|`, tức là **chia các viên đá thành hai nhóm** rồi lấy hiệu hai tổng; và cách chia tối ưu đạt được. Vậy đáp án là **hiệu nhỏ nhất** giữa tổng hai nhóm: bài toán **cái túi (knapsack / subset sum)** với sức chứa `tổng // 2`. Cách "đập hai viên nặng nhất" (như bài 1046) **sai** ở bài này.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề và **quy bài toán về chia hai nhóm** để hiệu hai tổng nhỏ nhất.
2. Nắm các công cụ Python cần thiết: `sum`, `itertools.product` (tập dấu `+/-`), mảng boolean quy hoạch động một chiều (duyệt **giảm dần** `j`), số nguyên lớn dùng làm **bitset** (`bits |= bits << w`), và `functools.lru_cache` cho mô phỏng trò chơi.
3. Hiểu nguyên lý: gọi `S` là tổng, tìm tổng con **lớn nhất** `A <= S // 2` đạt được bằng một tập con; đáp án là `S - 2 * A`.
4. Đi qua 4 cách: mô phỏng đúng trò chơi (oracle theo định nghĩa), liệt kê mọi tập dấu `+/-`, quy hoạch động cái túi (cách nộp chính) và bitset bằng số nguyên Python.
5. Tránh các lỗi kinh điển: **tham lam** đập hai viên nặng nhất, duyệt cái túi **tăng dần** (dùng một viên nhiều lần), trả `S - A` thay vì `S - 2 * A`, chọn tổng con **nhỏ nhất** thay vì lớn nhất `<= S // 2`, và mảng quy hoạch động thiếu một phần tử.

## 2. Đề bài & ví dụ

Cho mảng `stones` với `stones[i]` là khối lượng viên đá thứ `i`.

Trò chơi: mỗi lượt chọn **hai viên đá bất kỳ** và đập chúng vào nhau. Giả sử hai viên có khối lượng `x` và `y` với `x <= y`. Kết quả:

- nếu `x == y`: cả hai viên bị hủy;
- nếu `x != y`: viên khối lượng `x` bị hủy, viên khối lượng `y` có khối lượng mới `y - x`.

Cuối trò chơi còn **tối đa một viên**. Trả về **khối lượng nhỏ nhất** có thể có của viên còn lại. Nếu không còn viên nào, trả về `0`.

**Ví dụ 1:** `stones = [2,7,4,1,8,1]` -> `1`. Đập `2` và `4` được `2`: `[2,7,1,8,1]`; đập `7` và `8` được `1`: `[2,1,1,1]`; đập `2` và `1` được `1`: `[1,1,1]`; đập `1` và `1` được `0`: `[1]`.

**Ví dụ 2:** `stones = [31,26,33,21,40]` -> `5`.

**Ràng buộc:**

- `1 <= stones.length <= 30`
- `1 <= stones[i] <= 100`

In [1]:
from typing import List, Tuple
from functools import lru_cache
from itertools import product, combinations
import random, time

examples = [
    ([2, 7, 4, 1, 8, 1], 1),
    ([31, 26, 33, 21, 40], 5),
]
for stones, expected in examples:
    print(f"stones={stones} -> kỳ vọng {expected}")

stones=[2, 7, 4, 1, 8, 1] -> kỳ vọng 1
stones=[31, 26, 33, 21, 40] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Mỗi viên đá mang một dấu

Mỗi lần đập `x` và `y` (`x <= y`) cho `y - x`: một viên "mang dấu `+`", viên kia "dấu `-`". Lặp lại, viên cuối cùng có khối lượng **một tổ hợp** `±s_1 ± s_2 ± ... ± s_n` (mỗi viên xuất hiện đúng một lần với dấu `+` hoặc `-`), và vì khối lượng luôn không âm nên đó là giá trị tuyệt đối của tổng có dấu. Gọi `P` là tập các viên mang dấu `+` và `N` là tập còn lại: khối lượng cuối là `|sum(P) - sum(N)|`.

In [2]:
stones = [2, 7, 4, 1, 8, 1]
signs = [+1, -1, +1, -1, +1, -1]            # một cách gán dấu
print("tổng có dấu:", sum(s * w for s, w in zip(signs, stones)))
S = sum(stones)
A = sum(w for s, w in zip(signs, stones) if s > 0)
print(f"S = {S}, tổng nhóm '+' A = {A}, |A - (S - A)| = {abs(A - (S - A))} = |2A - S|")

tổng có dấu: 5
S = 23, tổng nhóm '+' A = 14, |A - (S - A)| = 5 = |2A - S|


### 3.2. Từ chia nhóm đến cái túi (subset sum)

Với một cách chia `(P, N)`, hiệu là `|S - 2 * sum(P)|` (vì `sum(N) = S - sum(P)`). Muốn hiệu **nhỏ nhất** ta chọn `sum(P)` **gần `S / 2` nhất**. Ta có thể giả sử `sum(P) <= S / 2` (đổi vai hai nhóm), nên cần **tổng con lớn nhất** `A <= S // 2` đạt được: đáp án `S - 2 * A`. Việc tìm tập các tổng con đạt được là bài toán **cái túi 0/1** (mỗi viên dùng tối đa một lần).

### 3.3. Cái túi 0/1 một chiều: vì sao duyệt giảm dần

`dp[j]` = "có tập con tổng đúng `j` không". Với mỗi viên `w`: `dp[j] |= dp[j - w]` cho `j` từ `target` **giảm dần** xuống `w`. Duyệt **giảm dần** bảo đảm `dp[j - w]` còn là giá trị **trước** khi xét viên `w` (chưa dùng `w`); duyệt tăng dần sẽ dùng `w` nhiều lần (cái túi **không giới hạn**).

In [3]:
def reachable_sums(stones, descending=True):
    target = sum(stones) // 2
    dp = [True] + [False] * target
    for w in stones:
        rng = range(target, w - 1, -1) if descending else range(w, target + 1)
        for j in rng:
            dp[j] = dp[j] or dp[j - w]
    return [j for j, ok in enumerate(dp) if ok]

print("giảm dần (đúng)  :", reachable_sums([3, 5]))
print("tăng dần (dùng lại viên đá):", reachable_sums([3, 5], descending=False))

giảm dần (đúng)  : [0, 3]
tăng dần (dùng lại viên đá): [0, 3]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `stones = [31, 26, 33, 21, 40]`, tổng `S = 151`, `S // 2 = 75`. Tổng con lớn nhất `<= 75`: ví dụ `33 + 40 = 73` (không có tập con nào có tổng `74` hoặc `75`, máy xác nhận ở các mục sau). Đáp án `151 - 2 * 73 = 5` ✓.

Ví dụ 1: `[2, 7, 4, 1, 8, 1]`, `S = 23`, `S // 2 = 11`. Tổng con lớn nhất `<= 11`: `11 = 7 + 4`: đáp án `23 - 22 = 1` ✓.

## 5. Cách 1 — Brute Force: mô phỏng đúng trò chơi

**Ý tưởng.** Làm đúng theo đề: trạng thái là **đa tập** các khối lượng (biểu diễn bằng bộ đã sắp xếp). Từ một trạng thái, thử **mọi cặp** hai viên để đập, sinh trạng thái mới; lấy khối lượng cuối nhỏ nhất. Dùng `lru_cache` để không tính lại. Không dùng ý tưởng chia nhóm nên đây là **oracle độc lập** (chỉ cho `n` nhỏ).

In [4]:
@lru_cache(None)
def play(state: Tuple[int, ...]) -> int:
    if len(state) == 0:
        return 0
    if len(state) == 1:
        return state[0]
    best = None
    for i in range(len(state)):
        for j in range(i + 1, len(state)):
            x, y = state[i], state[j]                       # x <= y vì state đã sắp xếp
            rest = list(state[:i] + state[i + 1:j] + state[j + 1:])
            if x != y:
                rest.append(y - x)
            val = play(tuple(sorted(rest)))
            if best is None or val < best:
                best = val
    return best

def stones_game(stones: List[int]) -> int:
    return play(tuple(sorted(stones)))

for stones, expected in examples[:1]:
    got = stones_game(stones)
    print(f"stones={stones}: mô phỏng = {got}", "OK" if got == expected else "SAI")
    assert got == expected
print("ví dụ 2 có 5 viên:", stones_game([31, 26, 33, 21, 40]))
assert stones_game([31, 26, 33, 21, 40]) == 5

stones=[2, 7, 4, 1, 8, 1]: mô phỏng = 1 OK
ví dụ 2 có 5 viên: 5


**Độ phức tạp và nút thắt.** Số trạng thái (đa tập kết quả sau các lượt đập) tăng rất nhanh theo `n`. Đo thử:

In [5]:
rng = random.Random(1049)
for n in (4, 5, 6, 7, 8):
    play.cache_clear()
    stones = [rng.randint(1, 100) for _ in range(n)]
    t0 = time.perf_counter()
    stones_game(stones)
    print(f"n={n}: {play.cache_info().currsize:8} trạng thái, {(time.perf_counter() - t0) * 1000:9.1f} ms")
print("n = 30: không thể")

n=4:       21 trạng thái,       0.0 ms
n=5:       92 trạng thái,       0.1 ms
n=6:      460 trạng thái,       1.0 ms
n=7:     1731 trạng thái,       5.4 ms
n=8:     4732 trạng thái,      17.9 ms
n = 30: không thể


Nút thắt: ta mô phỏng từng lượt đập, trong khi kết quả cuối chỉ phụ thuộc cách **gán dấu** `+/-` cho các viên. Ta chuyển sang bài toán chia nhóm.

## 6. Ý tưởng thuật toán / bước đột phá

**Đáp án `= min |sum(P) - sum(N)|` trên mọi cách chia `(P, N)`.**

- Mọi dãy các lượt đập cho một kết quả có dạng `|sum(P) - sum(N)|` (mục 3.1).
- Ngược lại, **cách chia tối ưu** luôn đạt được bằng một dãy lượt đập: ta chứng minh ở mục 10 bằng quy nạp (đập một viên của nhóm này với một viên của nhóm kia; hiệu không đổi và bài toán nhỏ đi).

Vậy bài toán là **chia hai nhóm với hiệu nhỏ nhất**, tức tìm tổng con lớn nhất `A <= S // 2` (cái túi 0/1); đáp án `S - 2 * A`.

| Cách | Ý tưởng |
|---|---|
| 2 | Liệt kê `2^n` tập dấu `+/-` (vét cạn bài toán chia nhóm) |
| 3 | Quy hoạch động cái túi `dp[j]` (nộp) |
| 4 | Bitset bằng số nguyên Python: `bits |= bits << w` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Liệt kê mọi tập dấu

**Giải thích từng đoạn.** Với mỗi `signs` thuộc `{+1, -1}^n` tính `abs(sum(s * w))` và lấy nhỏ nhất. `2^n` tập, mỗi tập `O(n)`: dùng cho `n` tới khoảng `18`.

In [6]:
def stones_signs(stones: List[int]) -> int:
    best = None
    for signs in product((1, -1), repeat=len(stones)):
        v = abs(sum(s * w for s, w in zip(signs, stones)))
        if best is None or v < best:
            best = v
    return best

for stones, expected in examples:
    got = stones_signs(stones)
    print(f"stones={stones}: tập dấu = {got}", "OK" if got == expected else "SAI")
    assert got == expected

stones=[2, 7, 4, 1, 8, 1]: tập dấu = 1 OK
stones=[31, 26, 33, 21, 40]: tập dấu = 5 OK


**Code nộp LeetCode (Cách 2):** (đúng nhưng **quá chậm** với `n = 30`: `2^30 ~ 10^9` tập; chỉ để minh họa)

In [7]:
class Solution:
    def lastStoneWeightII(self, stones: List[int]) -> int:
        from itertools import product
        return min(abs(sum(s * w for s, w in zip(signs, stones))) for signs in product((1, -1), repeat=len(stones)))

sol_signs = Solution().lastStoneWeightII
for stones, expected in examples:
    assert sol_signs(stones) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(2^n * n)` thời gian: đo thử:

In [8]:
for n in (10, 14, 16, 18):
    stones = [random.Random(n).randint(1, 100) for _ in range(n)]
    t0 = time.perf_counter()
    stones_signs(stones)
    dt = time.perf_counter() - t0
    print(f"n={n:2}: {dt * 1000:9.1f} ms -> ngoại suy n=30: ~{dt * 2 ** (30 - n) / 60:12.0f} phút")

n=10:       0.9 ms -> ngoại suy n=30: ~          16 phút
n=14:      18.6 ms -> ngoại suy n=30: ~          20 phút
n=16:      74.7 ms -> ngoại suy n=30: ~          20 phút


n=18:     330.6 ms -> ngoại suy n=30: ~          23 phút


---

### 7.2. Cách 3 — Quy hoạch động cái túi (cách nộp chính)

**Giải thích từng đoạn.**

- `total = sum(stones)`, `target = total // 2`.
- `dp = [True] + [False] * target`: `dp[j]` cho biết có tập con tổng đúng `j`.
- Với mỗi `w`: `for j in range(target, w - 1, -1): dp[j] = dp[j] or dp[j - w]` (duyệt **giảm dần**).
- `best` là `j` lớn nhất với `dp[j]` đúng; đáp án `total - 2 * best`.

In [9]:
def stones_dp(stones: List[int]) -> int:
    total = sum(stones)
    target = total // 2
    dp = [True] + [False] * target
    for w in stones:
        for j in range(target, w - 1, -1):
            dp[j] = dp[j] or dp[j - w]
    best = max(j for j in range(target + 1) if dp[j])
    return total - 2 * best

for stones, expected in examples:
    got = stones_dp(stones)
    print(f"stones={stones}: DP = {got}", "OK" if got == expected else "SAI")
    assert got == expected

stones=[2, 7, 4, 1, 8, 1]: DP = 1 OK
stones=[31, 26, 33, 21, 40]: DP = 5 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def lastStoneWeightII(self, stones: List[int]) -> int:
        total = sum(stones)
        target = total // 2
        dp = [True] + [False] * target              # dp[j]: có tập con tổng đúng j
        for w in stones:
            for j in range(target, w - 1, -1):      # giảm dần: mỗi viên đá dùng tối đa một lần
                dp[j] = dp[j] or dp[j - w]
        best = max(j for j in range(target + 1) if dp[j])    # tổng con lớn nhất <= total // 2
        return total - 2 * best

sol_dp = Solution().lastStoneWeightII
for stones, expected in examples:
    assert sol_dp(stones) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n * S / 2)` (`n = 30`, `S <= 3000`: `~ 45000` phép), bộ nhớ `O(S)`.

---

### 7.3. Cách 4 — Bitset bằng số nguyên Python

**Ý tưởng khác.** Dùng **một số nguyên lớn** làm bitset: bit `j` bật nghĩa là tổng `j` đạt được. Thêm viên `w`: `bits |= bits << w` (mọi tổng cũ cộng thêm `w`). Ban đầu `bits = 1` (tổng `0`). Cuối cùng duyệt `j` từ `S // 2` xuống tìm bit bật đầu tiên.

**Giải thích từng đoạn.** Phép dịch bit và `|` trên số nguyên lớn chạy rất nhanh (viết bằng C); không cần giới hạn `target` vì ta chỉ đọc các bit `<= S // 2`.

In [11]:
def stones_bitset(stones: List[int]) -> int:
    total = sum(stones)
    bits = 1
    for w in stones:
        bits |= bits << w
    j = total // 2
    while not (bits >> j) & 1:
        j -= 1
    return total - 2 * j

for stones, expected in examples:
    got = stones_bitset(stones)
    print(f"stones={stones}: bitset = {got}", "OK" if got == expected else "SAI")
    assert got == expected

stones=[2, 7, 4, 1, 8, 1]: bitset = 1 OK
stones=[31, 26, 33, 21, 40]: bitset = 5 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def lastStoneWeightII(self, stones: List[int]) -> int:
        total = sum(stones)
        bits = 1                                      # bit j bật: có tập con tổng j
        for w in stones:
            bits |= bits << w                         # thêm w vào mọi tổng đã có
        j = total // 2
        while not (bits >> j) & 1:                    # tổng con lớn nhất <= total // 2
            j -= 1
        return total - 2 * j

sol_bitset = Solution().lastStoneWeightII
for stones, expected in examples:
    assert sol_bitset(stones) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n * S / 64)` (phép toán trên từ máy 64 bit), bộ nhớ `O(S / 8)` byte.

## 8. TRACE từng bước

### 8.1. DP cái túi cho `[2, 7, 4, 1, 8, 1]`

In mảng `dp` (các `j` đạt được) sau mỗi viên đá, với `target = 11`.

In [13]:
def trace_dp(stones):
    total = sum(stones)
    target = total // 2
    dp = [True] + [False] * target
    print(f"total = {total}, target = {target}; dp ban đầu đạt được: {[j for j in range(target + 1) if dp[j]]}")
    for w in stones:
        for j in range(target, w - 1, -1):
            dp[j] = dp[j] or dp[j - w]
        print(f"sau viên {w}: đạt được {[j for j in range(target + 1) if dp[j]]}")
    best = max(j for j in range(target + 1) if dp[j])
    print(f"tổng con lớn nhất <= {target} là {best} -> đáp án {total} - 2 * {best} = {total - 2 * best}")
    return total - 2 * best

assert trace_dp([2, 7, 4, 1, 8, 1]) == 1

total = 23, target = 11; dp ban đầu đạt được: [0]
sau viên 2: đạt được [0, 2]
sau viên 7: đạt được [0, 2, 7, 9]
sau viên 4: đạt được [0, 2, 4, 6, 7, 9, 11]
sau viên 1: đạt được [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
sau viên 8: đạt được [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
sau viên 1: đạt được [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
tổng con lớn nhất <= 11 là 11 -> đáp án 23 - 2 * 11 = 1


### 8.2. Bitset cho `[3, 5, 7]`

In bitset dưới dạng nhị phân sau mỗi viên.

In [14]:
bits = 1
for w in [3, 5, 7]:
    bits |= bits << w
    print(f"sau viên {w}: {bits:b} -> tổng đạt được {[j for j in range(bits.bit_length()) if (bits >> j) & 1]}")
total = 15
j = total // 2
while not (bits >> j) & 1:
    j -= 1
print("tổng con lớn nhất <= 7:", j, "-> đáp án", total - 2 * j)
assert Solution().lastStoneWeightII([3, 5, 7]) == 1

sau viên 3: 1001 -> tổng đạt được [0, 3]
sau viên 5: 100101001 -> tổng đạt được [0, 3, 5, 8]
sau viên 7: 1001010110101001 -> tổng đạt được [0, 3, 5, 7, 8, 10, 12, 15]
tổng con lớn nhất <= 7: 7 -> đáp án 1


### 8.3. Vì sao tham lam (đập hai viên nặng nhất) sai

Với `[2, 7, 4, 1, 8, 1]`, đập hai viên nặng nhất lần lượt cho `1`. Nhưng với `[31, 26, 33, 21, 40]` thì cho kết quả lớn hơn `5`.

In [15]:
import heapq
def greedy_heaviest(stones):
    h = [-w for w in stones]
    heapq.heapify(h)
    while len(h) > 1:
        a, b = -heapq.heappop(h), -heapq.heappop(h)
        if a != b:
            heapq.heappush(h, -(a - b))
    return -h[0] if h else 0

print("tham lam:", greedy_heaviest([31, 26, 33, 21, 40]), "| đúng:", Solution().lastStoneWeightII([31, 26, 33, 21, 40]))
assert greedy_heaviest([31, 26, 33, 21, 40]) != 5

tham lam: 9 | đúng: 5


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (quy hoạch động cái túi): ngắn, `O(n * S)`, dễ chứng minh.

```python
class Solution:
    def lastStoneWeightII(self, stones: List[int]) -> int:
        total = sum(stones)
        target = total // 2
        dp = [True] + [False] * target              # dp[j]: có tập con tổng đúng j
        for w in stones:
            for j in range(target, w - 1, -1):      # giảm dần: mỗi viên đá dùng tối đa một lần
                dp[j] = dp[j] or dp[j - w]
        best = max(j for j in range(target + 1) if dp[j])    # tổng con lớn nhất <= total // 2
        return total - 2 * best
```

In [16]:
class Solution:
    def lastStoneWeightII(self, stones: List[int]) -> int:
        total = sum(stones)
        target = total // 2
        dp = [True] + [False] * target              # dp[j]: có tập con tổng đúng j
        for w in stones:
            for j in range(target, w - 1, -1):      # giảm dần: mỗi viên đá dùng tối đa một lần
                dp[j] = dp[j] or dp[j - w]
        best = max(j for j in range(target + 1) if dp[j])    # tổng con lớn nhất <= total // 2
        return total - 2 * best

sol = Solution()
for stones, expected in examples:
    got = sol.lastStoneWeightII(stones)
    print(f"stones={stones}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

stones=[2, 7, 4, 1, 8, 1]: 1 OK
stones=[31, 26, 33, 21, 40]: 5 OK


## 10. Vì sao đúng

**Kết quả có dạng chia nhóm (cận dưới).** Mỗi lượt đập thay hai viên `x <= y` bằng viên `y - x` (hoặc không viên nào nếu `x = y`): viên mới có thể xem là "viên `y` mang dấu `+`, viên `x` mang dấu `-`". Quy nạp: mọi viên còn lại ở bất kỳ thời điểm nào có khối lượng `|sum(P) - sum(N)|` với `P`, `N` là hai tập các viên **gốc** đã gộp vào nó, ngăn cách theo dấu. Khi kết thúc ta còn tối đa một viên, khối lượng của nó là `|sum(P) - sum(N)|` với `P ∪ N` là toàn bộ viên đá gốc (mọi viên đã hợp nhất). Do đó đáp án `>= min` trên mọi cách chia `(P, N)` của `|sum(P) - sum(N)|`.

**Cách chia tối ưu đạt được (cận trên).** Gọi `D` là hiệu nhỏ nhất trên mọi cách chia. Ta chứng minh quy nạp theo số viên `m` rằng trò chơi có thể kết thúc với khối lượng đúng `D`. Với `m = 1` thì `D` là khối lượng viên đó. Với `m >= 2`:

1. Tồn tại một cách chia tối ưu `(P, N)` với **cả hai nhóm khác rỗng**: nếu `N` rỗng thì hiệu là `S`, còn chuyển một viên `w` sang `N` cho hiệu `|S - 2w| < S` (vì `0 < w < S` khi còn viên khác), nên chia như vậy không tối ưu.
2. Lấy `a` thuộc `P` và `b` thuộc `N`, đập chúng: được viên `c = |a - b|` (hoặc không viên nào nếu `a = b`). Tập mới `M'` có `m - 1` (hoặc `m - 2`) viên và có một cách chia với **cùng hiệu** `D` (đặt `c` vào nhóm của viên lớn hơn). Vậy hiệu nhỏ nhất `D'` của `M'` thỏa `D' <= D`.
3. Ngược lại mọi cách chia `(X, Y)` của `M'` **nâng** được thành cách chia của `M` với **cùng hiệu** (thay `c` bởi `a` và `b`: viên lớn hơn vào cùng nhóm với `c`, viên nhỏ hơn vào nhóm còn lại; hiệu không đổi vì `c = |a - b|`). Nên `D' >= D`.

Suy ra `D' = D`; theo giả thiết quy nạp (hoặc khi `M'` rỗng thì `D = 0`) trò chơi trên `M'` có thể kết thúc với `D`, vậy trò chơi trên `M` cũng vậy. Kết hợp với cận dưới: đáp án đúng bằng `D`.

**Công thức cái túi.** Với tổng `S`, `sum(P) = A` thì hiệu là `|S - 2A|`; chọn `A <= S / 2` lớn nhất (đối xứng giữa hai nhóm) cho hiệu nhỏ nhất `S - 2A`. `dp[j]` đúng ⇔ tồn tại tập con tổng đúng `j` (duyệt giảm dần nên mỗi viên dùng một lần).

**Bitset.** `bits |= bits << w` thêm `w` vào mọi tổng đã có đúng một lần cho mỗi viên.

**Kiểm chứng bằng máy.** Oracle (Cách 1) mô phỏng chính trò chơi (mọi cặp, mọi thứ tự). Ta đối chiếu ba cách còn lại trên **mọi** mảng độ dài `<= 6` với giá trị `1..6`, và trên nhiều mảng ngẫu nhiên có seed; với `n` lớn (tới `30`) ta so quy hoạch động và bitset với một oracle độc lập khác: **gặp nhau ở giữa** (meet in the middle) trên hai nửa.

In [17]:
total = 0
for n in range(1, 7):
    for tup in product(range(1, 7), repeat=n):
        stones = list(tup)
        if tuple(sorted(stones)) != tup:
            continue                                   # chỉ cần các đa tập (thứ tự không quan trọng)
        ref = stones_game(stones)
        assert stones_signs(stones) == stones_dp(stones) == stones_bitset(stones) == sol.lastStoneWeightII(stones) == ref, stones
        total += 1
print("đã đối chiếu", total, "đa tập (n <= 6, giá trị 1..6): mô phỏng trò chơi = tập dấu = DP = bitset")

đã đối chiếu 923 đa tập (n <= 6, giá trị 1..6): mô phỏng trò chơi = tập dấu = DP = bitset


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng trò chơi | số trạng thái tăng rất nhanh | số trạng thái |
| 2. Tập dấu `+/-` | `O(2^n * n)` | `O(1)` |
| 3. DP cái túi | `O(n * S)` | `O(S)` |
| 4. Bitset | `O(n * S / 64)` | `O(S / 8)` byte |

Đo ở cỡ tối đa (`n = 30`, mỗi viên `<= 100`, `S <= 3000`):

In [18]:
def stones_meet_in_middle(stones: List[int]) -> int:
    """Oracle độc lập cho n lớn: tập tổng hiệu có dấu của hai nửa, tìm cặp có tổng gần 0 nhất."""
    from bisect import bisect_left
    half = len(stones) // 2
    def signed_sums(arr):
        sums = [0]
        for w in arr:
            sums = [s + w for s in sums] + [s - w for s in sums]
        return sums
    left, right = signed_sums(stones[:half]), sorted(signed_sums(stones[half:]))
    best = None
    for s in left:
        k = bisect_left(right, -s)
        for idx in (k - 1, k):
            if 0 <= idx < len(right):
                v = abs(s + right[idx])
                if best is None or v < best:
                    best = v
    return best

for stones, expected in examples:
    assert stones_meet_in_middle(stones) == expected

rng = random.Random(11)
cases = {"ngẫu nhiên 1..100": [rng.randint(1, 100) for _ in range(30)], "toàn 100": [100] * 30, "toàn 1": [1] * 30,
         "lũy thừa 2 (lệch)": [2 ** (i % 7) for i in range(30)], "rất lệch": [100] + [1] * 29}
for label, stones in cases.items():
    ref = stones_meet_in_middle(stones)
    row = [f"{label:20}"]
    for name, fn in (("DP", sol_dp), ("bitset", sol_bitset), ("nộp", sol.lastStoneWeightII)):
        t0 = time.perf_counter()
        got = fn(stones)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.3f} ms")
    print(" | ".join(row), "| đáp án", ref)

ngẫu nhiên 1..100    | DP   3.001 ms | bitset   0.024 ms | nộp   1.010 ms | đáp án 1
toàn 100             | DP   2.135 ms | bitset   0.026 ms | nộp   2.193 ms | đáp án 0


toàn 1               | DP   0.028 ms | bitset   0.006 ms | nộp   0.019 ms | đáp án 0
lũy thừa 2 (lệch)    | DP   0.255 ms | bitset   0.009 ms | nộp   0.229 ms | đáp án 1


rất lệch             | DP   0.085 ms | bitset   0.008 ms | nộp   0.087 ms | đáp án 71


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách (oracle mô phỏng trò chơi cho `n` nhỏ, gặp nhau ở giữa cho `n` lớn), test ngẫu nhiên có seed và các tính chất.

In [19]:
def all_solutions():
    return {"dp": stones_dp, "bitset": stones_bitset, "sol_dp": sol_dp, "sol_bitset": sol_bitset, "final": sol.lastStoneWeightII}

def check_all(stones, expected=None):
    results = {name: fn(list(stones)) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, (stones, results)
    ref = results["dp"]
    if len(stones) <= 7:
        assert stones_game(stones) == ref, (stones, stones_game(stones), ref)
    if len(stones) <= 16:
        assert stones_signs(stones) == ref
    assert stones_meet_in_middle(stones) == ref, stones
    if expected is not None:
        assert ref == expected, (stones, ref, expected)
    return ref

# 12.1 ví dụ đề
for stones, expected in examples:
    check_all(stones, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], 1),                                   # một viên: không đập được
    ([5], 5),
    ([1, 1], 0),                                # hai viên bằng nhau: cùng biến mất
    ([3, 7], 4),                                # hai viên: hiệu
    ([1, 2, 3], 0),                             # 1 + 2 = 3
    ([2, 2, 2], 2),                             # ba viên bằng nhau: còn một viên
    ([1, 1, 1, 1], 0),                          # chẵn viên bằng nhau
    ([100, 1], 99),                             # hiệu lớn
    ([3, 9, 7, 3], 2),                          # tính tay ở bài tập 2: {9, 3} và {7, 3}
    ([89, 61, 55, 83, 7], None),                # để oracle quyết định
    ([100] * 30, 0),                            # cỡ tối đa: 15 + 15
    ([100] * 29, 100),                          # số lẻ viên bằng nhau
    ([1] * 30, 0),
    ([100] + [1] * 29, 71),                     # một viên nặng và 29 viên nhẹ: 100 - 29
    ([31, 26, 33, 21, 40], 5),                  # ví dụ 2
]
for stones, expected in edge:
    check_all(stones, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK


12.2 edge case: OK 15 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle mô phỏng trò chơi)
def wrong_greedy(stones):                             # đập hai viên nặng nhất (bài 1046) áp dụng cho bài này
    return greedy_heaviest(stones)

def wrong_ascending_knapsack(stones):                 # duyệt tăng dần: dùng một viên nhiều lần
    total = sum(stones)
    target = total // 2
    dp = [True] + [False] * target
    for w in stones:
        for j in range(w, target + 1):
            dp[j] = dp[j] or dp[j - w]
    best = max(j for j in range(target + 1) if dp[j])
    return total - 2 * best

def wrong_total_minus_best(stones):                   # trả total - best thay vì total - 2 * best
    total = sum(stones)
    target = total // 2
    dp = [True] + [False] * target
    for w in stones:
        for j in range(target, w - 1, -1):
            dp[j] = dp[j] or dp[j - w]
    best = max(j for j in range(target + 1) if dp[j])
    return total - best

def wrong_smallest_sum(stones):                       # chọn tổng con NHỎ NHẤT > 0 thay vì lớn nhất <= total // 2
    total = sum(stones)
    target = total // 2
    dp = [True] + [False] * target
    for w in stones:
        for j in range(target, w - 1, -1):
            dp[j] = dp[j] or dp[j - w]
    best = min(j for j in range(1, target + 1) if dp[j]) if any(dp[1:]) else 0
    return total - 2 * best

def wrong_exact_half_only(stones):                    # chỉ xét chia đúng bằng nhau; không được thì trả tổng chẵn lẻ
    total = sum(stones)
    if total % 2:
        return 1
    target = total // 2
    dp = [True] + [False] * target
    for w in stones:
        for j in range(target, w - 1, -1):
            dp[j] = dp[j] or dp[j - w]
    return 0 if dp[target] else 2

def find_counter(wrong, vmax=8, max_n=6):
    for n in range(1, max_n + 1):
        for tup in product(range(1, vmax + 1), repeat=n):
            if tuple(sorted(tup)) != tup:
                continue
            if wrong(list(tup)) != stones_game(list(tup)):
                return list(tup)
    return None

counters = {}
for name, fn in (("tham lam đập hai viên nặng nhất", wrong_greedy), ("cái túi duyệt tăng dần", wrong_ascending_knapsack),
                 ("trả total - best (thiếu nhân 2)", wrong_total_minus_best), ("chọn tổng con nhỏ nhất", wrong_smallest_sum),
                 ("chỉ xét chia đúng bằng nhau", wrong_exact_half_only)):
    v = find_counter(fn)
    assert v is not None, name
    counters[name] = v
    print(f"{name:34} stones={v}: sai -> {fn(v)}, đúng -> {stones_game(v)}")
    check_all(v)
print("12.3 phản ví dụ: OK")

tham lam đập hai viên nặng nhất    stones=[2, 2, 2, 3, 3]: sai -> 2, đúng -> 0
cái túi duyệt tăng dần             stones=[1, 3]: sai -> 0, đúng -> 2
trả total - best (thiếu nhân 2)    stones=[1, 1]: sai -> 1, đúng -> 0
chọn tổng con nhỏ nhất             stones=[1, 1, 2]: sai -> 2, đúng -> 0
chỉ xét chia đúng bằng nhau        stones=[3]: sai -> 1, đúng -> 3
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên: nhiều miền giá trị; n nhỏ so với mô phỏng/tập dấu, n tới 30 so gặp nhau ở giữa
rng = random.Random(4049)
for _ in range(500):
    n = rng.randint(1, 8)
    check_all([rng.randint(1, rng.choice([3, 10, 100])) for _ in range(n)])
for _ in range(60):
    n = rng.randint(9, 30)
    check_all([rng.randint(1, rng.choice([5, 50, 100])) for _ in range(n)])
print("12.4 ngẫu nhiên: OK")

# 12.5 tính chất: cùng tính chẵn lẻ với tổng; <= max và <= tổng; bất biến theo hoán vị và nhân hệ số; đáp án 0 khi có chia đều; <= tổng nhỏ nhất của hai nhóm hiệu
rng = random.Random(5049)
for _ in range(400):
    n = rng.randint(1, 25)
    stones = [rng.randint(1, 100) for _ in range(n)]
    r = sol.lastStoneWeightII(stones)
    assert r % 2 == sum(stones) % 2                                  # hiệu hai nhóm cùng tính chẵn lẻ với tổng
    assert 0 <= r <= max(stones)                                     # luôn <= viên nặng nhất
    assert sol.lastStoneWeightII(random.Random(n).sample(stones, n)) == r
    assert sol.lastStoneWeightII([3 * w for w in stones]) == 3 * r   # nhân hệ số
    assert sol.lastStoneWeightII(stones + stones) == 0               # lặp đôi: chia đều được
print("12.5 tính chất: OK")

12.4 ngẫu nhiên: OK


12.5 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: đập hai viên nặng nhất

Cách này đúng cho bài **Last Stone Weight (1046)** nơi luôn chọn hai viên nặng nhất; bài này cho **chọn bất kỳ hai viên**, đáp án nhỏ hơn nhiều.

In [22]:
v = counters["tham lam đập hai viên nặng nhất"]
print("stones =", v, "| sai ->", wrong_greedy(v), "| đúng ->", Solution().lastStoneWeightII(v))
assert wrong_greedy(v) != Solution().lastStoneWeightII(v)

stones = [2, 2, 2, 3, 3] | sai -> 2 | đúng -> 0


Vì sao sai: kết quả tối ưu cần chia hai nhóm cân bằng, không phải luôn đập hai viên to nhất. **Đúng:** cái túi.

### Lỗi 2 — Cái túi duyệt **tăng dần**

Duyệt `j` tăng dần cho phép dùng cùng một viên nhiều lần (cái túi không giới hạn).

In [23]:
v = counters["cái túi duyệt tăng dần"]
print("stones =", v, "| sai ->", wrong_ascending_knapsack(v), "| đúng ->", Solution().lastStoneWeightII(v))
assert wrong_ascending_knapsack(v) != Solution().lastStoneWeightII(v)

stones = [1, 3] | sai -> 0 | đúng -> 2


Vì sao sai: `dp[j]` có thể dựa vào `dp[j - w]` đã cập nhật **trong cùng lượt `w`**. **Đúng:** `range(target, w - 1, -1)`.

### Lỗi 3 — Trả `total - best` (thiếu nhân 2)

Hiệu hai nhóm là `(total - best) - best = total - 2 * best`.

In [24]:
v = counters["trả total - best (thiếu nhân 2)"]
print("stones =", v, "| sai ->", wrong_total_minus_best(v), "| đúng ->", Solution().lastStoneWeightII(v))
assert wrong_total_minus_best(v) != Solution().lastStoneWeightII(v)

stones = [1, 1] | sai -> 1 | đúng -> 0


Vì sao sai: `total - best` là tổng nhóm kia, không phải hiệu. **Đúng:** `total - 2 * best`.

### Lỗi 4 — Chọn tổng con **nhỏ nhất** thay vì lớn nhất `<= total // 2`

Muốn hai nhóm cân bằng nhất cần tổng gần `total / 2` nhất.

In [25]:
v = counters["chọn tổng con nhỏ nhất"]
print("stones =", v, "| sai ->", wrong_smallest_sum(v), "| đúng ->", Solution().lastStoneWeightII(v))
assert wrong_smallest_sum(v) != Solution().lastStoneWeightII(v)

stones = [1, 1, 2] | sai -> 2 | đúng -> 0


Vì sao sai: tổng con nhỏ nhất làm hiệu lớn. **Đúng:** `max(j for j in range(target + 1) if dp[j])`.

### Lỗi 5 — Chỉ xét chia đúng bằng nhau

Không phải lúc nào cũng chia đều được; đáp án có thể là hiệu nhỏ nhất bất kỳ (`1`, `2`, `5`, ...), không chỉ `0` hoặc `1`.

In [26]:
v = counters["chỉ xét chia đúng bằng nhau"]
print("stones =", v, "| sai ->", wrong_exact_half_only(v), "| đúng ->", Solution().lastStoneWeightII(v))
assert wrong_exact_half_only(v) != Solution().lastStoneWeightII(v)
print("ví dụ 2: sai ->", wrong_exact_half_only([31, 26, 33, 21, 40]), "(đúng 5)")

stones = [3] | sai -> 1 | đúng -> 3
ví dụ 2: sai -> 1 (đúng 5)


Vì sao sai: ví dụ 2 cho `5`. **Đúng:** tìm tổng con lớn nhất `<= total // 2`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "ghép các phần tử bằng phép trừ/hiệu", "kết quả cuối có thể là `±a ± b ± ...`" ⇒ quy về **chia hai nhóm**, tổng con gần `S / 2` nhất ⇒ **cái túi 0/1**.
- Cái túi 0/1 một chiều: duyệt sức chứa **giảm dần**; bitset Python `bits |= bits << w` rất gọn và nhanh.
- Đáp án `= S - 2 * A` với `A` là tổng con lớn nhất `<= S // 2`.
- Đừng áp dụng tham lam của bài "Last Stone Weight" (chọn hai viên nặng nhất) khi được **chọn bất kỳ hai viên**.
- Cùng họ: Partition Equal Subset Sum (416), Target Sum (494), Last Stone Weight (1046), Partition Array Into Two Arrays to Minimize Sum Difference (2035).
- Test nên có: một viên, hai viên bằng nhau, tất cả bằng nhau, một viên rất nặng, `n = 30`.

## 15. Biến thể

1. **Chia đều?** (416): tồn tại tập con tổng `S / 2` không: `dp[S // 2]` khi `S` chẵn.
2. **Số cách gán dấu để tổng bằng mục tiêu** (494): đếm số tập con.
3. **Khi `n` lớn nhưng tổng nhỏ** hoặc **`n` nhỏ nhưng tổng lớn**: dùng bitset hoặc gặp nhau ở giữa (`2^(n/2)`).
4. Bài liên quan: 416, 494, 1046.

Biến thể 1 (chia đều) kiểm bằng oracle tập dấu:

In [27]:
def can_partition(nums: List[int]) -> bool:
    total = sum(nums)
    if total % 2:
        return False
    bits = 1
    for w in nums:
        bits |= bits << w
    return bool((bits >> (total // 2)) & 1)

def can_partition_brute(nums):
    return any(sum(s * w for s, w in zip(signs, nums)) == 0 for signs in product((1, -1), repeat=len(nums)))

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(1, 12) for _ in range(rng.randint(1, 12))]
    assert can_partition(nums) == can_partition_brute(nums), nums
print(can_partition([1, 5, 11, 5]), can_partition([1, 2, 3, 5]), "(ví dụ của bài 416: True False)")

True False (ví dụ của bài 416: True False)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng trò chơi | Theo đề | rất lớn | rất lớn | Oracle, `n <= 8` |
| 2. Tập dấu `+/-` | Quy về chia nhóm | `O(2^n * n)` | `O(1)` | Oracle, `n <= 18` |
| 3. DP cái túi | `dp[j]` giảm dần | `O(n * S)` | `O(S)` | **Cách nộp chính** |
| 4. Bitset | `bits \|= bits << w` | `O(n * S / 64)` | `O(S / 8)` | Gọn, nhanh |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(reachable_sums([2, 3]), reachable_sums([2, 3], descending=False))
print(Solution().lastStoneWeightII([5]), Solution().lastStoneWeightII([4, 4]), Solution().lastStoneWeightII([1, 2, 4]))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # hai danh sách
my_guess_2 = None      # ba số

### Bài 2 — Tính tay (dễ-vừa)

`stones = [3, 9, 7, 3]`. Ghi đáp án.

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cái túi.

In [30]:
def exercise_3(stones: List[int]) -> int:
    total = sum(stones)
    target = total // 2
    dp = [True] + [False] * target
    for w in stones:
        # TODO 1: duyệt j từ target GIẢM DẦN xuống w; dp[j] = dp[j] or dp[j - w]
        pass
    # TODO 2: best = j lớn nhất có dp[j]
    # TODO 3: trả về total - 2 * best
    pass


# for stones, expected in examples:
#     assert exercise_3(stones) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(stones)` bằng **bitset** (số nguyên Python).

In [31]:
def exercise_4(stones: List[int]) -> int:
    # TODO
    pass


# for stones, expected in examples:
#     assert exercise_4(stones) == expected

### Bài 5 — Mini challenge: chỉ ra cách chia (khó)

Viết `exercise_5(stones)` trả về **danh sách chỉ số** của một nhóm `A` sao cho `|sum(A) - sum(phần còn lại)|` **bằng đáp án** (nhỏ nhất). Có nhiều đáp án đúng; ta kiểm tra bằng **hàm kiểm tra**.

```python
exercise_5([2, 7, 4, 1, 8, 1])     -> ví dụ [1, 2] (7 + 4 = 11 và phần còn lại 12: hiệu 1)
exercise_5([1, 1])                 -> ví dụ [0] (hiệu 0)
```
Gợi ý: lưu cách đạt từng tổng (ví dụ `choice[j]` = chỉ số viên cuối cùng thêm vào) hoặc truy vết bảng 2 chiều.

In [32]:
def exercise_5(stones: List[int]) -> List[int]:
    # TODO
    pass


def check_partition(stones, idx) -> bool:
    idx = list(idx)
    if len(set(idx)) != len(idx) or any(not (0 <= i < len(stones)) for i in idx):
        return False
    a = sum(stones[i] for i in idx)
    return abs(a - (sum(stones) - a)) == Solution().lastStoneWeightII(stones)


# for stones, _ in examples:
#     assert check_partition(stones, exercise_5(stones))

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `reachable_sums` trả các tổng con đạt được `<= S // 2`.
- Mức 2: `[2, 3]`: `S // 2 = 2`: các tổng `0, 2`; duyệt tăng dần cũng cho cùng vì `target` nhỏ.
- Mức 3: dòng 1: `[0, 2] [0, 2]`; dòng 2: `5 0 1` (`[1, 2, 4]`: `1 + 2` và `4` hiệu `1`).

**Bài 2**
- Mức 1: tổng `S = 22`, `target = 11`.
- Mức 2: tổng con lớn nhất `<= 11`: `3 + 7 = 10` (không có `11`).
- Mức 3: `22 - 2 * 10 = 2`.

**Bài 3**
- Mức 1: `for j in range(target, w - 1, -1): dp[j] = dp[j] or dp[j - w]`.
- Mức 2: `best = max(j for j in range(target + 1) if dp[j])`.
- Mức 3: `return total - 2 * best`.

**Bài 4**
- Mức 1: `bits = 1`; mỗi `w`: `bits |= bits << w`.
- Mức 2: `j = total // 2`; giảm `j` cho tới khi `(bits >> j) & 1`.
- Mức 3: `return total - 2 * j`.

**Bài 5**
- Mức 1: dùng bảng `dp[i][j]` (hai chiều) hoặc lưu `prev[j]` khi bật `dp[j]` lần đầu.
- Mức 2: truy vết từ `best`: nếu `dp[i-1][j]` đúng thì viên `i - 1` không dùng, ngược lại dùng và `j -= stones[i - 1]`.
- Mức 3: trả danh sách các chỉ số đã dùng.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(reachable_sums([2, 3]), reachable_sums([2, 3], descending=False))
print(Solution().lastStoneWeightII([5]), Solution().lastStoneWeightII([4, 4]), Solution().lastStoneWeightII([1, 2, 4]))

assert (reachable_sums([2, 3]), reachable_sums([2, 3], descending=False)) == ([0, 2], [0, 2])
assert (Solution().lastStoneWeightII([5]), Solution().lastStoneWeightII([4, 4]), Solution().lastStoneWeightII([1, 2, 4])) == (5, 0, 1)

[0, 2] [0, 2]
5 0 1


### Đáp án Bài 2

`S = 22`, `S // 2 = 11`. Các tổng con: `10 = 3 + 7` là lớn nhất `<= 11` (không có `11`). Đáp án `22 - 20 = 2`.

In [34]:
hand_answer_solution = 2
assert Solution().lastStoneWeightII([3, 9, 7, 3]) == hand_answer_solution == stones_game([3, 9, 7, 3])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(stones: List[int]) -> int:
    total = sum(stones)
    target = total // 2
    dp = [True] + [False] * target
    for w in stones:
        for j in range(target, w - 1, -1):
            dp[j] = dp[j] or dp[j - w]
    best = max(j for j in range(target + 1) if dp[j])
    return total - 2 * best

for stones, expected in examples:
    assert exercise_3_solution(stones) == expected
rng = random.Random(3)
for _ in range(300):
    stones = [rng.randint(1, 20) for _ in range(rng.randint(1, 7))]
    assert exercise_3_solution(stones) == stones_game(stones)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(stones: List[int]) -> int:
    total = sum(stones)
    bits = 1
    for w in stones:
        bits |= bits << w
    j = total // 2
    while not (bits >> j) & 1:
        j -= 1
    return total - 2 * j

for stones, expected in examples:
    assert exercise_4_solution(stones) == expected
rng = random.Random(4)
for _ in range(300):
    stones = [rng.randint(1, 20) for _ in range(rng.randint(1, 7))]
    assert exercise_4_solution(stones) == stones_game(stones)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(stones: List[int]) -> List[int]:
    n = len(stones)
    total = sum(stones)
    target = total // 2
    dp = [[False] * (target + 1) for _ in range(n + 1)]
    dp[0][0] = True
    for i in range(1, n + 1):
        w = stones[i - 1]
        for j in range(target + 1):
            dp[i][j] = dp[i - 1][j] or (j >= w and dp[i - 1][j - w])
    j = max(j for j in range(target + 1) if dp[n][j])
    idx = []
    for i in range(n, 0, -1):
        if dp[i - 1][j]:
            continue                                   # viên i - 1 không cần
        idx.append(i - 1)
        j -= stones[i - 1]
    return idx

for stones, _ in examples:
    assert check_partition(stones, exercise_5_solution(stones))
rng = random.Random(5)
for _ in range(300):
    stones = [rng.randint(1, 30) for _ in range(rng.randint(1, 15))]
    assert check_partition(stones, exercise_5_solution(stones)), stones
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (cái túi 0/1, chia hai nhóm)

```python
class Solution:
    def lastStoneWeightII(self, stones: List[int]) -> int:
        total = sum(stones)
        target = total // 2
        dp = [True] + [False] * target              # dp[j]: có tập con tổng đúng j
        for w in stones:
            for j in range(target, w - 1, -1):      # giảm dần: mỗi viên đá dùng tối đa một lần
                dp[j] = dp[j] or dp[j - w]
        best = max(j for j in range(target + 1) if dp[j])    # tổng con lớn nhất <= total // 2
        return total - 2 * best
```

Công thức nhớ nhanh: đáp án `= S - 2 * A`, `A` là tổng con lớn nhất `<= S // 2` (cái túi 0/1).

### Checklist trước khi submit

- [ ] Nhận ra bài toán là **chia hai nhóm** (không phải tham lam đập hai viên nặng nhất)?
- [ ] Cái túi duyệt sức chứa **giảm dần**?
- [ ] Mảng `dp` có `target + 1` phần tử (`target = total // 2`)?
- [ ] Chọn tổng con **lớn nhất** `<= total // 2`, trả `total - 2 * best`?
- [ ] Đã thử: một viên, hai viên bằng nhau, toàn bằng nhau, một viên rất nặng, `n = 30`?